# 8.1 Forecasting Service — Demand Forecasting (XGBoost)

**Goal:** Predict **tomorrow's demand** per product (SKU) and generate a **Preparation Order (T+1)**: `(product_id, quantity)`.

This notebook avoids common pitfalls:
- **No leakage** (features use only information available up to day *t*)
- **No empty splits** (we trim inactive tails safely + split on active dates, with guards)
- Warehouse-friendly evaluation (overall + **active demand only**)


In [15]:
import numpy as np
import pandas as pd
from pathlib import Path

# LightGBM (Kaggle usually has it; fallback install if missing)
try:
    import lightgbm as lgb
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "lightgbm"])
    import lightgbm as lgb


## 1) Load demand history (Kaggle)

We load the demand history file that contains:
- `date` (or a date column)
- `id_produit`
- `quantite_demande`


In [16]:
ROOT = Path('/kaggle/input')

def find_file(filename: str) -> Path:
    hits = list(ROOT.rglob(filename)) if ROOT.exists() else []
    if not hits:
        raise FileNotFoundError(f"Couldn't find {filename} under {ROOT}. Add the dataset to this notebook.")
    return hits[0]

hist_path = find_file('WMS_Hackathon_DataPack_Templates_FR_FV_B7_ONLY_historique_demande.xlsx')
df_historique_demande = pd.read_excel(hist_path, engine='openpyxl')

print('Loaded:', hist_path)
print('Columns:', list(df_historique_demande.columns))
df_historique_demande.head()

Loaded: /kaggle/input/datasets/haithembouziane/data14/WMS_Hackathon_DataPack_Templates_FR_FV_B7_ONLY_historique_demande.xlsx
Columns: ['date', 'id_produit', 'quantite_demande']


,date,id_produit,quantite_demande
0,2024-03-09 10:20:17,31779,4
1,2024-03-09 10:20:17,31775,1
2,2024-03-09 10:31:09,34013,7
3,2024-03-09 10:31:09,34014,1
4,2024-03-09 10:31:09,34016,4


## 2) Clean & normalize demand table

Key fixes vs the previous version:
- **Do NOT use `.dropna()` inside column assignment** (it breaks indices and can wipe data).
- Normalize dates to **day-level** with `.dt.normalize()` so you get a real daily series.


In [17]:
def clean_id(series: pd.Series) -> pd.Series:
    out = series.astype(str).str.strip()
    out = out.replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    out = out.str.replace(r"\.0$", "", regex=True)   # remove trailing .0
    return out

demand = df_historique_demande.copy()

# If your file uses different column names, adjust them here:
# expected: 'date', 'id_produit', 'quantite_demande'
if "date" not in demand.columns:
    raise KeyError("Column 'date' not found. Check df_historique_demande.columns and rename accordingly.")
if "id_produit" not in demand.columns:
    raise KeyError("Column 'id_produit' not found. Check df_historique_demande.columns and rename accordingly.")
if "quantite_demande" not in demand.columns:
    raise KeyError("Column 'quantite_demande' not found. Check df_historique_demande.columns and rename accordingly.")

demand["date"] = pd.to_datetime(demand["date"], errors="coerce").dt.normalize()  # day-level
demand["id_produit"] = clean_id(demand["id_produit"])
demand["qty_demand"] = pd.to_numeric(demand["quantite_demande"], errors="coerce")

# Drop invalid key rows
demand = demand.dropna(subset=["date","id_produit"])
# Missing qty -> treat as 0
demand["qty_demand"] = demand["qty_demand"].fillna(0).astype(int)

demand_daily = (demand.groupby(["date","id_produit"], as_index=False)
                      .agg(qty_demand=("qty_demand","sum"))
               )

print("demand_daily rows:", len(demand_daily))
print("date range:", demand_daily["date"].min(), "->", demand_daily["date"].max())
print("unique dates:", demand_daily["date"].nunique())
print("unique products:", demand_daily["id_produit"].nunique())

# Guard: need at least 2 days to build y_tomorrow
if demand_daily["date"].nunique() < 2:
    raise ValueError(
        "Only 1 unique day found in demand_daily. "
        "This means your 'date' column isn't providing a time series. "
        "Verify the correct date column in the Excel file."
    )

demand_daily.head()

demand_daily rows: 80659
date range: 2024-01-02 00:00:00 -> 2026-01-08 00:00:00
unique dates: 598
unique products: 1129


,date,id_produit,qty_demand
0,2024-01-02,31759,100
1,2024-01-02,31769,560
2,2024-01-03,31357,50
3,2024-01-03,31371,25
4,2024-01-03,31373,865


## 3) Build a complete daily grid per product (fill missing with 0)

We also drop products that never have demand (sum=0). If this would remove everything, we skip the filter.


In [18]:
sku_total = demand_daily.groupby("id_produit")["qty_demand"].sum()
active_skus = sku_total[sku_total > 0].index

if len(active_skus) == 0:
    print("WARNING: No active SKUs found (all sums are 0). Skipping SKU filtering.")
else:
    demand_daily = demand_daily[demand_daily["id_produit"].isin(active_skus)].copy()

min_date, max_date = demand_daily["date"].min(), demand_daily["date"].max()
all_dates = pd.date_range(min_date, max_date, freq="D")
all_products = demand_daily["id_produit"].unique()

idx = pd.MultiIndex.from_product([all_dates, all_products], names=["date","id_produit"])
df = (demand_daily.set_index(["date","id_produit"])
                .reindex(idx, fill_value=0)
                .reset_index()
                .sort_values(["id_produit","date"])
                .reset_index(drop=True))

print("Full grid rows:", len(df))
print("Full grid unique dates:", df["date"].nunique())
df.head()

Full grid rows: 833202
Full grid unique dates: 738


,date,id_produit,qty_demand
0,2024-01-02,31334,0
1,2024-01-03,31334,0
2,2024-01-04,31334,0
3,2024-01-05,31334,0
4,2024-01-06,31334,0


## 4) Trim inactive tail safely

We trim to the last day where demand > 0 **only if such a day exists**.


In [19]:
last_active_date = df.loc[df["qty_demand"] > 0, "date"].max()

if pd.isna(last_active_date):
    print("WARNING: No positive demand found; skipping tail trimming.")
else:
    df = df[df["date"] <= last_active_date].copy()
    print("Trimmed max date:", df["date"].max())

print("Last 14 days total demand:")
print(df.groupby("date")["qty_demand"].sum().tail(14))

Trimmed max date: 2026-01-08 00:00:00
Last 14 days total demand:
date
2025-12-26         0
2025-12-27    144724
2025-12-28     44107
2025-12-29    274312
2025-12-30      7941
2025-12-31       146
2026-01-01         0
2026-01-02         0
2026-01-03    132839
2026-01-04    176986
2026-01-05     27500
2026-01-06    141346
2026-01-07    239923
2026-01-08     30512
Name: qty_demand, dtype: int64


## 5) Create label = tomorrow demand

In [20]:
df = df.sort_values(["id_produit","date"]).reset_index(drop=True)
df["y_tomorrow"] = df.groupby("id_produit")["qty_demand"].shift(-1)

df_model = df.dropna(subset=["y_tomorrow"]).copy()

print("df rows:", len(df))
print("df_model rows:", len(df_model))
print("df date min/max:", df["date"].min(), "->", df["date"].max())
print("df_model date min/max:", df_model["date"].min(), "->", df_model["date"].max())
print("positives in qty_demand:", int((df["qty_demand"] > 0).sum()))
print("positives in y_tomorrow:", int((df_model["y_tomorrow"] > 0).sum()))

if len(df_model) == 0:
    raise ValueError("df_model is empty. Usually this happens when there is only 1 unique date. Check cell 2 date parsing.")

df_model.head()

df rows: 833202
df_model rows: 832073
df date min/max: 2024-01-02 00:00:00 -> 2026-01-08 00:00:00
df_model date min/max: 2024-01-02 00:00:00 -> 2026-01-07 00:00:00
positives in qty_demand: 80659
positives in y_tomorrow: 80657


,date,id_produit,qty_demand,y_tomorrow
0,2024-01-02,31334,0,0.0
1,2024-01-03,31334,0,0.0
2,2024-01-04,31334,0,0.0
3,2024-01-05,31334,0,0.0
4,2024-01-06,31334,0,544.0


## 6) Feature engineering (no leakage)

In [21]:
g = df_model.groupby("id_produit", group_keys=False)

# Lags
for lag in [1, 2, 3, 7, 14, 28]:
    df_model[f"d_lag{lag}"] = g["qty_demand"].shift(lag).fillna(0)

# Rolling stats (shift(1) avoids leakage)
def rmean(s, w): return s.shift(1).rolling(w, min_periods=1).mean()
def rsum(s, w):  return s.shift(1).rolling(w, min_periods=1).sum()
def rstd(s, w):  return s.shift(1).rolling(w, min_periods=2).std().fillna(0)
def rmax(s, w):  return s.shift(1).rolling(w, min_periods=1).max()

for w in [7, 14, 28]:
    df_model[f"d_mean{w}"] = g["qty_demand"].transform(lambda s: rmean(s, w))
    df_model[f"d_sum{w}"]  = g["qty_demand"].transform(lambda s: rsum(s, w))
    df_model[f"d_std{w}"]  = g["qty_demand"].transform(lambda s: rstd(s, w))
    df_model[f"d_max{w}"]  = g["qty_demand"].transform(lambda s: rmax(s, w))

# Intermittency: zero rate in last 28 days (shifted)
df_model["zero_rate_28"] = g["qty_demand"].transform(
    lambda s: s.shift(1).rolling(28, min_periods=1).apply(lambda x: (np.array(x)==0).mean(), raw=True)
).fillna(1.0)

# Days since last non-zero (shifted by 1 day)
def days_since_nonzero(series: pd.Series) -> pd.Series:
    last = -10**9
    out = np.empty(len(series), dtype=int)
    arr = series.to_numpy()
    for i, v in enumerate(arr):
        if v > 0:
            last = i
        out[i] = (i - last) if last > -10**8 else 9999
    return pd.Series(out, index=series.index)

ds = g["qty_demand"].apply(days_since_nonzero)
df_model["days_since_nz"] = ds.groupby(df_model["id_produit"]).shift(1).fillna(9999)

# Trend / momentum
df_model["trend_7_28"] = df_model["d_mean7"] - df_model["d_mean28"]
df_model["momentum_1_7"] = df_model["d_lag1"] - df_model["d_mean7"]

# Calendar features
dt = df_model["date"]
df_model["dow"] = dt.dt.dayofweek
df_model["is_weekend"] = (df_model["dow"] >= 5).astype(int)
df_model["month"] = dt.dt.month
df_model["weekofyear"] = dt.dt.isocalendar().week.astype(int)

# Product frequency encoding
freq = df_model["id_produit"].value_counts()
df_model["pid_freq"] = df_model["id_produit"].map(freq).astype(float)

df_model.head()

,date,id_produit,qty_demand,y_tomorrow,d_lag1,d_lag2,d_lag3,d_lag7,d_lag14,d_lag28,...,d_max28,zero_rate_28,days_since_nz,trend_7_28,momentum_1_7,dow,is_weekend,month,weekofyear,pid_freq
0,2024-01-02,31334,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,1.000000,9999.0,NaN,NaN,1,0,1,1,737.0
1,2024-01-03,31334,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.500000,9999.0,0.0,0.0,2,0,1,1,737.0
2,2024-01-04,31334,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.666667,9999.0,0.0,0.0,3,0,1,1,737.0
3,2024-01-05,31334,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.750000,9999.0,0.0,0.0,4,0,1,1,737.0
4,2024-01-06,31334,0,544.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.800000,9999.0,0.0,0.0,5,1,1,1,737.0


## 7) Robust time split (active dates + guards)

In [22]:
daily_total = df_model.groupby("date")["y_tomorrow"].sum()
active_dates = daily_total[daily_total > 0].index.sort_values()
all_dates_unique = np.array(sorted(df_model["date"].unique()))

if len(all_dates_unique) < 5:
    raise ValueError(f"Not enough unique dates to split (found {len(all_dates_unique)}). Need at least ~5 days.")

if len(active_dates) >= 30:
    cut1 = active_dates[int(0.70 * len(active_dates))]
    cut2 = active_dates[int(0.85 * len(active_dates))]
else:
    cut1 = all_dates_unique[int(0.70 * len(all_dates_unique))]
    cut2 = all_dates_unique[int(0.85 * len(all_dates_unique))]

train = df_model[df_model["date"] <= cut1].copy()
val   = df_model[(df_model["date"] > cut1) & (df_model["date"] <= cut2)].copy()
test  = df_model[df_model["date"] > cut2].copy()

print("Cut1:", cut1, "Cut2:", cut2)
print("TRAIN positives:", int((train["y_tomorrow"] > 0).sum()))
print("VAL positives  :", int((val["y_tomorrow"] > 0).sum()))
print("TEST positives :", int((test["y_tomorrow"] > 0).sum()))
print("VAL dates:", val["date"].nunique(), "TEST dates:", test["date"].nunique())

Cut1: 2025-06-10 00:00:00 Cut2: 2025-09-23 00:00:00
TRAIN positives: 55863
VAL positives  : 12817
TEST positives : 11977
VAL dates: 105 TEST dates: 106


## 8) Train XGBoost (Poisson) + evaluation

In [28]:
target = "y_tomorrow"
feature_cols = [c for c in df_model.columns if c.startswith(("d_","zero_rate","days_since","trend","momentum"))] + \
               ["dow","is_weekend","month","weekofyear","pid_freq"]

X_train, y_train = train[feature_cols], train[target]
X_val, y_val     = val[feature_cols], val[target]
X_test, y_test   = test[feature_cols], test[target]

train_set = lgb.Dataset(X_train, label=y_train)
val_set   = lgb.Dataset(X_val, label=y_val, reference=train_set)

params = {
    "objective": "poisson",
    "metric": ["mae"],
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 50,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "seed": 42,
    "verbosity": -1,
}

callbacks = [
    lgb.early_stopping(stopping_rounds=150, verbose=True),
    lgb.log_evaluation(period=100),
]

model = lgb.train(
    params=params,
    train_set=train_set,
    num_boost_round=8000,
    valid_sets=[train_set, val_set],
    valid_names=["train", "val"],
    callbacks=callbacks,
)

pred_val  = np.clip(model.predict(X_val,  num_iteration=model.best_iteration),  0, None)
pred_test = np.clip(model.predict(X_test, num_iteration=model.best_iteration), 0, None)

def mae(y, p):
    y = np.asarray(y); p = np.asarray(p)
    return float(np.mean(np.abs(y - p)))

def rmse(y, p):
    y = np.asarray(y); p = np.asarray(p)
    return float(np.sqrt(np.mean((y - p) ** 2)))

def wmape(y, p):
    y = np.asarray(y); p = np.asarray(p)
    denom = np.sum(np.abs(y))
    return float(np.sum(np.abs(y - p)) / (denom if denom != 0 else 1.0))

def smape(y, p):
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    denom = np.abs(y) + np.abs(p)
    denom = np.where(denom == 0, 1.0, denom)
    return float(np.mean(2.0 * np.abs(p - y) / denom))

print("\nVAL (overall)")
print("MAE  :", mae(y_val.values, pred_val))
print("RMSE :", rmse(y_val.values, pred_val))
print("wMAPE:", wmape(y_val.values, pred_val))
print("sMAPE:", smape(y_val.values, pred_val))

print("\nTEST (overall)")
print("MAE  :", mae(y_test.values, pred_test))
print("RMSE :", rmse(y_test.values, pred_test))
print("wMAPE:", wmape(y_test.values, pred_test))
print("sMAPE:", smape(y_test.values, pred_test))

# Active-only metrics (optional)
val_mask  = (y_val.values > 0)
test_mask = (y_test.values > 0)

print("\nActive rows:")
print("VAL active:", int(val_mask.sum()), "/", len(y_val))
print("TEST active:", int(test_mask.sum()), "/", len(y_test))

if val_mask.any():
    print("\nVAL (active demand only)")
    print("MAE  :", mae(y_val.values[val_mask], pred_val[val_mask]))
    print("RMSE :", rmse(y_val.values[val_mask], pred_val[val_mask]))
    print("wMAPE:", wmape(y_val.values[val_mask], pred_val[val_mask]))
    print("sMAPE:", smape(y_val.values[val_mask], pred_val[val_mask]))

if test_mask.any():
    print("\nTEST (active demand only)")
    print("MAE  :", mae(y_test.values[test_mask], pred_test[test_mask]))
    print("RMSE :", rmse(y_test.values[test_mask], pred_test[test_mask]))
    print("wMAPE:", wmape(y_test.values[test_mask], pred_test[test_mask]))
    print("sMAPE:", smape(y_test.values[test_mask], pred_test[test_mask]))

# Baseline: lag1 (overall)
print("\nBaseline overall wMAPE:")
print("VAL lag1:", wmape(y_val.values,  val["d_lag1"].values))
print("TEST lag1:", wmape(y_test.values, test["d_lag1"].values))


Training until validation scores don't improve for 150 rounds
[100]	train's l1: 57.1333	val's l1: 68.2894
[200]	train's l1: 52.0312	val's l1: 65.7195
[300]	train's l1: 49.7161	val's l1: 65.1501
[400]	train's l1: 47.9884	val's l1: 64.8925
[500]	train's l1: 46.5719	val's l1: 64.7846
[600]	train's l1: 45.4425	val's l1: 64.682
[700]	train's l1: 44.4189	val's l1: 64.5246
[800]	train's l1: 43.4821	val's l1: 64.403
[900]	train's l1: 42.6616	val's l1: 64.2755
[1000]	train's l1: 41.8842	val's l1: 64.1249
[1100]	train's l1: 41.1123	val's l1: 63.9919
[1200]	train's l1: 40.4911	val's l1: 63.9431
[1300]	train's l1: 39.9077	val's l1: 63.8355
[1400]	train's l1: 39.3224	val's l1: 63.7595
[1500]	train's l1: 38.7673	val's l1: 63.6252
[1600]	train's l1: 38.2717	val's l1: 63.5798
[1700]	train's l1: 37.7804	val's l1: 63.5358
[1800]	train's l1: 37.3178	val's l1: 63.5029
[1900]	train's l1: 36.8877	val's l1: 63.4954
[2000]	train's l1: 36.4624	val's l1: 63.4168
[2100]	train's l1: 36.079	val's l1: 63.319
[2200]

## 9) Generate tomorrow's Preparation Order (forecasted demand)

In [29]:
today = df_model["date"].max()
tomorrow = today + pd.Timedelta(days=1)

X_today = df_model[df_model["date"] == today][feature_cols]
pred_tomorrow = np.clip(model.predict(X_today, num_iteration=model.best_iteration), 0, None)

prep = df_model[df_model["date"] == today][["id_produit"]].copy()
prep["date_preparation"] = tomorrow
prep["pred_qty_demand"] = pred_tomorrow
prep["qty_to_prepare"] = np.maximum(0, np.round(prep["pred_qty_demand"])).astype(int)

prep = prep[prep["qty_to_prepare"] > 0].sort_values("qty_to_prepare", ascending=False)

print("Tomorrow:", tomorrow.date())
prep.head(30)


Tomorrow: 2026-01-08


,id_produit,date_preparation,pred_qty_demand,qty_to_prepare
239848,31732,2026-01-08,4118.493375,4118
234682,31725,2026-01-08,4057.922085,4058
146860,31565,2026-01-08,3895.775913,3896
141694,31557,2026-01-08,3066.281522,3066
233206,31723,2026-01-08,2705.243607,2705
139480,31554,2026-01-08,2189.693507,2190
499624,34016,2026-01-08,2164.404329,2164
233944,31724,2026-01-08,2109.960828,2110
498886,34015,2026-01-08,1988.681155,1989
137266,31551,2026-01-08,1856.357822,1856
